# Task 03 -- Direction preservation under independent (u,v) Gaussianization (`docs/open_questions.md` §2.4)

`open_questions.md` §2.4 is now PARTIALLY MOOT for magnitude (resolved by
§2.3's `log E` tensor) but the **angle** question -- does independent
per-cell Gaussianization of `u_wsea`/`v_wsea` distort
`atan2(v,u)` -- remains open. See `tasks/03-direction-gaussianization-angle-preservation.md`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2025, 12, 1, tzinfo=timezone.utc)

from train.directional import ComingFromToUV

X_raw = load_training_window(DATA_DIR, T_START, T_END, ["wsh", "wsd"])
# wsd confirmed "coming_from" (open_questions.md §1.2).
X_uv = ComingFromToUV(pairs=[("wsh", "wsd", "u_wsea", "v_wsea")]).fit_transform(X_raw)

ocean_mask = (
    ~np.isnan(X_uv["u_wsea"]).any(dim="time").values
    & ~np.isnan(X_uv["v_wsea"]).any(dim="time").values
)
print(f"{ocean_mask.mean():.1%} ocean")


## Reference angle, before any Gaussianization

In [ ]:
u_true = X_uv["u_wsea"].values  # (T, n_lat, n_lon)
v_true = X_uv["v_wsea"].values
theta_true = np.degrees(np.arctan2(u_true, v_true)) % 360.0  # (u, v) arg order, per §2.1


## Independently Gaussianize u and v per cell, then recompute the angle

In [ ]:
from train.copula import GaussianCopula

uv_ds = X_uv[["u_wsea", "v_wsea"]]
gaussianized = GaussianCopula(n_bins=500).fit(uv_ds)
Z = gaussianized.transform(uv_ds)
uv_reconstructed = gaussianized.inverse_transform(Z)  # round-trip: identity if exact

u_rt = uv_reconstructed["u_wsea"].values
v_rt = uv_reconstructed["v_wsea"].values
theta_rt = np.degrees(np.arctan2(u_rt, v_rt)) % 360.0

angular_error = np.abs(((theta_rt - theta_true + 180.0) % 360.0) - 180.0)  # min circular distance
valid = ocean_mask[None, :, :] & ~np.isnan(angular_error)
print(f"round-trip angular error: mean {np.nanmean(angular_error[valid]):.3f} deg, "
      f"p99 {np.nanpercentile(angular_error[valid], 99):.3f} deg")


## Decision-rule diagnostic

The round-trip check above isolates the copula's own (Gaussianize + invert)
distortion. The open question is really about a *forecast* round-trip:
Gaussianize -> sample/perturb in Z-space -> invert -> compare angle to what a
directly-decomposed direction would have given. Extending this cell to draw
`Z` perturbations (not just invert the exact fit) and comparing against a
`hypot`-free direction reference is the next step -- record findings in
`tasks/03-direction-gaussianization-angle-preservation.md` §4.